# 04 - Análisis de Interpretabilidad del Modelo

Análisis de explicabilidad del clasificador de intenciones usando **LIME** (Local Interpretable Model-agnostic Explanations).

## Objetivos:
1. Entender qué palabras influyen más en las predicciones
2. Visualizar la importancia de características
3. Analizar patrones de comportamiento del modelo
4. Validar coherencia con el problema de negocio

## 1. Importar librerías

In [ ]:
import spacy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lime.lime_text import LimeTextExplainer
from sklearn.model_selection import train_test_split
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Librerías cargadas")

## 2. Cargar modelo y datos

In [ ]:
# Cargar modelo entrenado
model_path = Path("../models/intent_classifier")
nlp = spacy.load(model_path)

# Obtener etiquetas
class_names = sorted(list(nlp.get_pipe("textcat_multilabel").labels))

print(f"✅ Modelo cargado desde: {model_path}")
print(f"🏷️ Clases ({len(class_names)}): {class_names}")

In [ ]:
# Cargar datos
data = []
with open('../data/intents.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

texts = [item['text'] for item in data]
labels = [item['label'] for item in data]

# Dividir datos (misma semilla que entrenamiento)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"📊 Datos cargados: {len(data)} total, {len(test_texts)} para análisis")

## 3. Descripción del Modelo

### Arquitectura
- **Base**: spaCy con modelo de lenguaje español `es_core_news_sm`
- **Componente**: `textcat_multilabel` (TextCategorizer)
- **Arquitectura interna**: Ensemble de bag-of-words + CNN
- **Tipo**: Clasificación multiclase (22 intents)

### Objetivo del Modelo
Clasificar mensajes de texto de usuarios de WhatsApp para determinar su intención y proporcionar respuestas automáticas relevantes en el contexto de un restaurante de comida típica.

In [ ]:
# Información del modelo
print("=" * 60)
print("📋 DESCRIPCIÓN DEL MODELO")
print("=" * 60)
print(f"\n🔧 Pipeline: {nlp.pipe_names}")
print(f"📊 Número de clases: {len(class_names)}")
print(f"📝 Ejemplos de entrenamiento: {len(train_texts)}")
print(f"🧪 Ejemplos de prueba: {len(test_texts)}")
print(f"\n🎯 Objetivo: Clasificar intenciones de usuarios de WhatsApp")
print(f"🏢 Contexto: Chatbot para restaurante Los Motes de la Magdalena")

## 4. Configurar LIME para Explicabilidad

In [ ]:
def predict_proba(texts):
    """
    Función de predicción compatible con LIME.
    Retorna matriz de probabilidades [n_samples, n_classes]
    """
    results = []
    for text in texts:
        doc = nlp(text)
        # Obtener probabilidades en el orden de class_names
        probs = [doc.cats.get(label, 0.0) for label in class_names]
        results.append(probs)
    return np.array(results)

# Crear explicador LIME para texto
explainer = LimeTextExplainer(
    class_names=class_names,
    split_expression=r'\W+',  # Dividir por palabras
    random_state=42
)

print("✅ Explicador LIME configurado")

## 5. Análisis de Casos Individuales con LIME

In [ ]:
# Seleccionar ejemplos representativos de diferentes intents
ejemplos_analisis = [
    "hola buenos días",
    "cuál es el horario de atención",
    "quiero hacer un pedido",
    "hacen envíos a domicilio",
    "cuánto cuesta el mote de queso",
    "dónde están ubicados",
    "gracias por la atención",
    "quiero hablar con una persona"
]

print("🔍 Ejemplos seleccionados para análisis:")
for i, texto in enumerate(ejemplos_analisis, 1):
    doc = nlp(texto)
    pred = max(doc.cats, key=doc.cats.get)
    conf = doc.cats[pred]
    print(f"   {i}. \"{texto}\" → {pred} ({conf:.1%})")

In [ ]:
# Generar explicaciones LIME para cada ejemplo
explicaciones = []

print("\n📊 EXPLICACIONES LIME POR EJEMPLO")
print("=" * 70)

for texto in ejemplos_analisis:
    # Obtener predicción
    doc = nlp(texto)
    pred_label = max(doc.cats, key=doc.cats.get)
    pred_idx = class_names.index(pred_label)
    
    # Generar explicación LIME
    exp = explainer.explain_instance(
        texto,
        predict_proba,
        num_features=10,
        num_samples=500,
        labels=[pred_idx]
    )
    
    explicaciones.append({
        'texto': texto,
        'prediccion': pred_label,
        'confianza': doc.cats[pred_label],
        'explicacion': exp,
        'palabras_importantes': exp.as_list(label=pred_idx)
    })
    
    print(f"\n📝 Texto: \"{texto}\"")
    print(f"   Predicción: {pred_label} ({doc.cats[pred_label]:.1%})")
    print(f"   Palabras más influyentes:")
    for palabra, peso in exp.as_list(label=pred_idx)[:5]:
        signo = "+" if peso > 0 else ""
        print(f"      • '{palabra}': {signo}{peso:.4f}")

## 6. Visualización de Importancia de Palabras

In [ ]:
# Visualizar explicación para el primer ejemplo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, exp_data in enumerate(explicaciones[:4]):
    ax = axes[i]
    palabras_pesos = exp_data['palabras_importantes'][:6]
    
    if palabras_pesos:
        palabras = [p[0] for p in palabras_pesos]
        pesos = [p[1] for p in palabras_pesos]
        colores = ['green' if p > 0 else 'red' for p in pesos]
        
        ax.barh(palabras, pesos, color=colores)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        ax.set_xlabel('Peso (contribución a la predicción)')
        ax.set_title(f'"{exp_data["texto"][:30]}..."\n→ {exp_data["prediccion"]}')

plt.suptitle('Importancia de Palabras según LIME', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/lime_palabras_importantes.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Gráfico guardado en: reports/lime_palabras_importantes.png")

## 7. Análisis Global: Palabras Más Influyentes por Intent

In [ ]:
# Analizar múltiples ejemplos por intent para encontrar patrones
from collections import defaultdict

# Agrupar textos por intent
textos_por_intent = defaultdict(list)
for texto, label in zip(test_texts, test_labels):
    textos_por_intent[label].append(texto)

# Analizar palabras importantes por intent (muestreo)
palabras_por_intent = defaultdict(list)
intents_a_analizar = ['saludo', 'consultar_horario', 'realizar_pedido', 'delivery', 'despedida']

print("🔍 Analizando patrones por intent...")

for intent in intents_a_analizar:
    textos_intent = textos_por_intent[intent][:3]  # Máximo 3 por velocidad
    
    for texto in textos_intent:
        try:
            doc = nlp(texto)
            pred_label = max(doc.cats, key=doc.cats.get)
            pred_idx = class_names.index(pred_label)
            
            exp = explainer.explain_instance(
                texto,
                predict_proba,
                num_features=5,
                num_samples=300,
                labels=[pred_idx]
            )
            
            for palabra, peso in exp.as_list(label=pred_idx):
                if peso > 0:
                    palabras_por_intent[intent].append((palabra, peso))
        except:
            continue
    
    print(f"   ✅ {intent}: analizado")

print("\n✅ Análisis completado")

In [ ]:
# Mostrar palabras más frecuentes por intent
print("\n📋 PALABRAS CLAVE POR INTENT")
print("=" * 60)

resumen_palabras = {}

for intent in intents_a_analizar:
    palabras = palabras_por_intent[intent]
    if palabras:
        # Agregar pesos por palabra
        palabra_peso = defaultdict(float)
        for palabra, peso in palabras:
            palabra_peso[palabra.lower()] += peso
        
        # Top 5 palabras
        top_palabras = sorted(palabra_peso.items(), key=lambda x: x[1], reverse=True)[:5]
        resumen_palabras[intent] = top_palabras
        
        print(f"\n🏷️ {intent.upper()}:")
        for palabra, peso in top_palabras:
            print(f"   • {palabra}: {peso:.4f}")

## 8. Análisis de Patrones de Comportamiento

In [ ]:
# Analizar distribución de confianza por intent
confianzas_por_intent = defaultdict(list)

for texto, label_real in zip(test_texts, test_labels):
    doc = nlp(texto)
    pred_label = max(doc.cats, key=doc.cats.get)
    confianza = doc.cats[pred_label]
    confianzas_por_intent[label_real].append(confianza)

# Calcular estadísticas
stats_confianza = []
for intent in sorted(confianzas_por_intent.keys()):
    confs = confianzas_por_intent[intent]
    stats_confianza.append({
        'Intent': intent,
        'Media': np.mean(confs),
        'Min': np.min(confs),
        'Max': np.max(confs),
        'Ejemplos': len(confs)
    })

df_stats = pd.DataFrame(stats_confianza).sort_values('Media', ascending=False)
print("📊 CONFIANZA PROMEDIO POR INTENT")
print("=" * 60)
display(df_stats)

In [ ]:
# Visualizar distribución de confianza
fig, ax = plt.subplots(figsize=(12, 8))

intents_ordenados = df_stats['Intent'].tolist()
medias = df_stats['Media'].tolist()
colores = ['green' if m > 0.8 else 'orange' if m > 0.6 else 'red' for m in medias]

bars = ax.barh(intents_ordenados, medias, color=colores)
ax.axvline(x=0.8, color='green', linestyle='--', alpha=0.5, label='Alta confianza (0.8)')
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Umbral mínimo (0.5)')

for bar, val in zip(bars, medias):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')

ax.set_xlabel('Confianza Promedio')
ax.set_ylabel('Intent')
ax.set_title('Confianza Promedio del Modelo por Intent')
ax.set_xlim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/confianza_por_intent.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Reflexión: Coherencia con el Problema Organizacional

In [ ]:
print("=" * 70)
print("📋 REFLEXIÓN: COHERENCIA CON EL PROBLEMA ORGANIZACIONAL")
print("=" * 70)

print("""
🎯 PROBLEMA ORGANIZACIONAL:
   Los Motes de la Magdalena necesita automatizar la atención al cliente
   en WhatsApp para responder preguntas frecuentes y facilitar pedidos.

✅ COHERENCIA DEL MODELO:

1. PALABRAS CLAVE DETECTADAS:
   • "horario", "hora", "abren" → consultar_horario ✓
   • "pedir", "pedido", "quiero" → realizar_pedido ✓
   • "delivery", "domicilio", "envío" → delivery ✓
   • "hola", "buenos días" → saludo ✓
   
   El modelo captura correctamente el vocabulario del dominio.

2. INTENTS DE ALTA CONFIANZA:
   Los saludos y despedidas tienen alta confianza porque son
   expresiones cortas y distintivas.

3. INTENTS DE MENOR CONFIANZA:
   Algunos intents como 'fallback' tienen menor confianza,
   lo cual es esperado ya que capturan mensajes ambiguos.

4. VALOR PARA EL NEGOCIO:
   • Reduce carga de trabajo del personal
   • Respuestas instantáneas 24/7
   • Captura de intents de pedido para conversión
   • Escalamiento a humano cuando es necesario (hablar_humano)

📊 CONCLUSIÓN:
   El modelo es coherente con el problema organizacional.
   Las palabras que influyen en las predicciones son relevantes
   para el contexto de un restaurante y atención al cliente.
""")

## 10. Resumen de Hallazgos

In [ ]:
print("=" * 70)
print("📋 RESUMEN DE HALLAZGOS - ANÁLISIS DE INTERPRETABILIDAD")
print("=" * 70)

print("""
🔬 TÉCNICA UTILIZADA: LIME (Local Interpretable Model-agnostic Explanations)

📊 VARIABLES MÁS INFLUYENTES POR INTENT:
""")

for intent, palabras in resumen_palabras.items():
    print(f"   • {intent}: {', '.join([p[0] for p in palabras[:3]])}")

print("""
📈 PATRONES OBSERVADOS:
   1. Las palabras interrogativas ("cuál", "dónde", "cuánto") activan
      intents de consulta.
   2. Palabras de acción ("pedir", "quiero") activan intents transaccionales.
   3. Saludos formales e informales son bien diferenciados.
   4. El contexto "domicilio"/"delivery" es distintivo.

✅ FORTALEZAS DEL MODELO:
   • Buena separación entre intents de consulta y transaccionales
   • Alta confianza en intents comunes (saludo, despedida)
   • Palabras clave del dominio son correctamente identificadas

⚠️ ÁREAS DE MEJORA:
   • Algunos intents con pocos ejemplos tienen menor confianza
   • Mensajes muy cortos pueden ser ambiguos
   • Considerar sinónimos regionales adicionales
""")

print("=" * 70)
print("Fin del análisis de interpretabilidad")
print("=" * 70)